# Trim the user's message before your expensive model reads it

You run a travel chat bot. Every reply comes from `gpt-5.6-sol`, because planning a trip needs
real reasoning and Sol is the model that does it well. Sol also costs $5 per million input
tokens, and it re-reads the whole conversation on every turn.

The problem is what people actually type. An excited first-timer thinks out loud and circles back
to the same wishes in different words. That repetition is input Sol pays for, again on every turn
it stays in the chat.

[Alexandria](https://github.com/ucsc-cse115a-alexandria/alexandria) spots the sentences that make
a point an earlier one already made and folds them into one, keeping the meaning. The merge runs
on `gpt-5.6-luna` at $1 per million input tokens, one fifth of Sol's input price. So you spend a
little on the cheap model once to shrink what the expensive model reads every turn.

> Needs Python 3.14, [uv](https://docs.astral.sh/uv/), and `OPENAI_API_KEY` (a `.env` file
> works). Install the CLI with
> `uv tool install "git+https://github.com/ucsc-cse115a-alexandria/alexandria.git"`.
> The bot cells also need `openai` and `python-dotenv`. Every cell below makes real calls to
> `gpt-5.6-sol` and `gpt-5.6-luna`; the cost is a few cents.

## 1. A real message, with all the padding

Here is the kind of message an excited first-timer sends. Thinking out loud, they come back to
art, walking, budget, and crowds a second time, each in slightly different words. Every fact
matters; the second pass is what we can trim.

In [ ]:
!cat user_message.txt

## 2. Reduce it on the cheap model

Real messages rarely repeat a sentence word for word; they restate the same point in new words.
Those soft duplicates sit around 0.7 cosine similarity, below Alexandria's cautious default of
0.85, so we pass `--threshold 0.7` to catch paraphrase. `-v` streams each pair it found and the
single line `gpt-5.6-luna` merged them into, so you can see exactly what was trimmed. Section 3
confirms the meaning held.

In [ ]:
!alexandria reduce user_message.txt --threshold 0.7 -v > reduced.txt
!cat reduced.txt

## 3. Did it keep the meaning?

A lower threshold merges more, so the guard matters more. `compare` embeds both versions and
reports how close they stayed next to how much shorter the message got. Add `--min-similarity 0.9`
and it exits non-zero in CI when a reduction drifts too far.

In [ ]:
!alexandria compare user_message.txt reduced.txt | tee compare.json

## 4. What Sol pays

Sol re-reads the message on every turn it stays in the context window, so the saving recurs. The
reduction itself was a single pass on Luna, the $1/MTok model, paid once. The longer the chat
runs, the more that one-time trim pays back.

In [ ]:
import json
from pathlib import Path

report = json.loads(Path("compare.json").read_text())

SOL_INPUT_PER_TOKEN = 5 / 1_000_000  # gpt-5.6-sol input: $5 / MTok
saved_tokens = report["original_tokens"] - report["edited_tokens"]
saved_per_turn = saved_tokens * SOL_INPUT_PER_TOKEN

print(f"tokens dropped: {saved_tokens}\n")
print("Sol input saved, by how long the message stays in context:")
for turns in (1, 5, 20):
    print(f"  over {turns:>2} turns: ${saved_per_turn * turns:.5f}")
print("\nThe trim was one pass on gpt-5.6-luna ($1/MTok in), paid once.")

## 5. Does the shorter message change the trip?

Section 3 was the cheap embedding gate. This is the end-to-end check. The bot writes a plan from
each version on Sol, then an LLM referee reads the original message and both plans and rules
whether they deliver the same trip. The referee runs on Luna, since judging is cheap. If the
reduction had dropped a wish, the two plans would diverge and this would fail.

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # OPENAI_API_KEY from a .env file or the environment
client = OpenAI()

BOT_MODEL = "gpt-5.6-sol"  # the expensive reasoning model that writes the plan
SYSTEM = "You are a friendly travel planner. Reply with a short day-by-day itinerary."

raw_message = Path("user_message.txt").read_text()
reduced_message = Path("reduced.txt").read_text()


def plan_trip(message: str) -> str:
    reply = client.chat.completions.create(
        model=BOT_MODEL,
        reasoning_effort="medium",
        seed=7,  # best-effort reproducibility so reruns give the same-ish plan
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": message},
        ],
    )
    return reply.choices[0].message.content.strip()


from_full = plan_trip(raw_message)
from_short = plan_trip(reduced_message)

print("--- plan from the full message ---\n" + from_full[:600])
print("\n--- plan from the shortened message ---\n" + from_short[:600])

In [ ]:
import json

REFEREE_MODEL = "gpt-5.6-luna"  # judging is cheap, so the referee uses the cheap model
REFEREE = (
    "You check whether shortening a traveler's message changed the trip they get. You are given "
    "the traveler's ORIGINAL message and two itineraries: one written from the original, one from "
    "a shortened copy. Decide whether both itineraries serve the ORIGINAL request equally well, "
    "with nothing the traveler asked for dropped by the shorter one. Ignore wording, length, and "
    'which exact sights each picks. Reply as JSON: {"same_trip": true or false, "reason": "<one sentence>"}.'
)


def referee(original: str, plan_a: str, plan_b: str) -> dict:
    reply = client.chat.completions.create(
        model=REFEREE_MODEL,
        reasoning_effort="low",
        seed=7,  # best-effort reproducibility so the verdict is stable across runs
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": REFEREE},
            {
                "role": "user",
                "content": (
                    f"Original message:\n{original}\n\n"
                    f"Itinerary A (from the original):\n{plan_a}\n\n"
                    f"Itinerary B (from the shortened message):\n{plan_b}"
                ),
            },
        ],
    )
    return json.loads(reply.choices[0].message.content)


verdict = referee(raw_message, from_full, from_short)

print(f"LLM referee ({REFEREE_MODEL})")
print("same trip:", "PASS" if verdict["same_trip"] else "FAIL")
print("reason:", verdict["reason"])

## Takeaways

- A natural, rambly message got about a third shorter with the same trip intact.
- Natural repeats are paraphrases, not copies, so lower the merge threshold (`--threshold 0.7`)
  and let `compare` guard the meaning.
- The merge ran on `gpt-5.6-luna` ($1/MTok in); the saving lands on `gpt-5.6-sol` ($5/MTok in),
  which re-reads the message every turn, so one trim pays back across the whole chat.
- One command does it: `alexandria reduce user_message.txt --threshold 0.7 > reduced.txt`.